In [ ]:
import pandas as pd
import numpy as np

# ==== File Paths ====
disbursement_file = r"Input_Files/Sheet1.xlsx"
ytd_file = r"Input_Files/8.Disbursement Fagun 2082.xlsx"
main_file = r"Input_Files/Duelist 19th Feb, 2026.xlsx"
output_file = r"Output_Files/updated_disbursement.xlsx"

# =====================================================
# ✅ STEP 1 — READ FILES
# =====================================================
disb_df = pd.read_excel(disbursement_file, dtype=str)

ytd_df = pd.read_excel(
    ytd_file,
    sheet_name='YTD',
    dtype=str
)

main_df = pd.read_excel(
    main_file,
    sheet_name='Mainsheet',
    dtype=str
)

# =====================================================
# ✅ STEP 2 — CLEAN COLUMN NAMES
# =====================================================
def clean_columns(df):

    df.columns = df.columns.str.strip()

    # Remove duplicate columns
    df = df.loc[:, ~df.columns.duplicated()]

    return df.astype(str)

disb_df = clean_columns(disb_df)
ytd_df = clean_columns(ytd_df)
main_df = clean_columns(main_df)

# =====================================================
# ✅ STEP 3 — STANDARDIZE COLUMN NAMES
# =====================================================
def standardize(df):

    rename_dict = {}

    for col in df.columns:

        key = col.replace(" ", "").lower()

        # AcType
        if key in ["actype", "at"]:
            rename_dict[col] = "AcType"

        # Loan Type
        if key in ["loantype", "oldacnum"]:
            rename_dict[col] = "Loan Type"

        # Branch
        if key in ["branchname", "branch"]:
            rename_dict[col] = "BranchName"

    df = df.rename(columns=rename_dict)

    # Remove duplicate columns
    df = df.loc[:, ~df.columns.duplicated()]

    return df

disb_df = standardize(disb_df)
ytd_df = standardize(ytd_df)
main_df = standardize(main_df)

# =====================================================
# ✅ STEP 4 — CLEAN VALUES
# =====================================================
for df in [disb_df, ytd_df, main_df]:

    for col in ["AcType", "Loan Type", "BranchName"]:

        if col in df.columns:

            df[col] = (
                df[col]
                .astype(str)
                .str.strip()
                .replace(["", "nan", "None"], np.nan)
            )

# Remove AcType 4Z
disb_df = disb_df[
    disb_df["AcType"] != "4Z"
]

# =====================================================
# ✅ STEP 5 — FIRST CHECK IN DUELIST
# =====================================================

main_map = main_df[
    main_df["Loan Type"].notna()
][["AcType", "BranchName", "Loan Type"]].drop_duplicates()

merged_df = disb_df.merge(
    main_map,
    on=["AcType", "BranchName"],
    how="left",
    suffixes=("", "_main")
)

# Fill blank loan type from Duelist
merged_df["Loan Type"] = merged_df["Loan Type"].fillna(
    merged_df["Loan Type_main"]
)

# Drop extra column
merged_df.drop(
    columns=["Loan Type_main"],
    inplace=True
)

# =====================================================
# ✅ STEP 6 — SECOND CHECK IN YTD
# =====================================================

ytd_map = ytd_df[
    ytd_df["Loan Type"].notna()
][["AcType", "BranchName", "Loan Type"]].drop_duplicates()

merged_df = merged_df.merge(
    ytd_map,
    on=["AcType", "BranchName"],
    how="left",
    suffixes=("", "_ytd")
)

# Fill remaining blank loan type from YTD
merged_df["Loan Type"] = merged_df["Loan Type"].fillna(
    merged_df["Loan Type_ytd"]
)

# Drop extra column
merged_df.drop(
    columns=["Loan Type_ytd"],
    inplace=True
)

# =====================================================
# ✅ STEP 7 — EXPORT RESULT
# =====================================================
merged_df.to_excel(output_file, index=False)

print("✅ Loan Type mapping completed!")
print("📁 Output saved to:", output_file)

# =====================================================
# ✅ STEP 8 — SHOW UNMATCHED
# =====================================================
unmatched = merged_df[
    merged_df["Loan Type"].isna()
]

print("⚠ Unmatched rows:", len(unmatched))